# Train ViT + GPT-2 (best config) on the full training set

This notebook trains the winning architecture from the fine-tune comparison — **ViT encoder + GPT-2 decoder** — with the exact hyperparameters that scored best:

| learning_rate | weight_decay | dropout | freeze_gpt2_base |
|---------------|--------------|---------|------------------|
| 1e-4          | 0.01         | 0.2     | True             |

It uses the **same training recipe** as the comparison notebook — full training set each epoch, **15 epochs with early stopping** (monitor val BLEU-4, patience 3), per-epoch best-weight checkpointing and rollback — and produces the **same visualizations** (validation BLEU-4 per epoch and training loss per epoch). Encoder is a frozen feature extractor; only cross-attention + the encoder projection are trained (freeze_gpt2_base=True). Built for Colab (also runs on a local Jupyter).

## 1. Install dependencies and imports

In [ ]:
# Install once if needed
!pip -q install transformers pycocotools nltk

import os, json, random, time, copy
from dataclasses import dataclass, asdict
from typing import Optional
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoConfig, AutoModel, AutoModelForCausalLM, AutoTokenizer, AutoImageProcessor,
)

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Using device:", device)

## 2. Project settings

In [ ]:
# Portable paths: works on Colab AND a local Jupyter notebook.
try:
    import google.colab  # noqa: F401
    BASE_DIR = "/content"
except ImportError:
    BASE_DIR = os.path.abspath(".")

DATA_DIR = os.path.join(BASE_DIR, "data", "coco")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs_vit_gpt2")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

COCO_SPLIT = "val"      # "val" for a smaller project run; "train" for full COCO if downloaded
FAST_DEV_RUN = False    # True only while debugging

# ---- Training budget (same recipe as the fine-tune comparison) --------------
EPOCHS = 1 if FAST_DEV_RUN else 15                 # full training length
EARLY_STOP_PATIENCE = 3
EARLY_STOP_MIN_DELTA = 1e-3
FULL_TRAIN_MAX_BATCHES = 3 if FAST_DEV_RUN else None  # None = full train set each epoch

# ---- Shared knobs -----------------------------------------------------------
BATCH_SIZE = 32
EVAL_NUM_BATCHES = 3 if FAST_DEV_RUN else 10       # val batches used for BLEU
MAX_TEXT_LEN = 40
MAX_GEN_LEN = 40
NUM_WORKERS = 2

# ---- Winning ViT + GPT-2 hyperparameters (from the fine-tune comparison) -----
ENCODER_NAME = "google/vit-base-patch16-224-in21k"
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01
DROPOUT = 0.2
FREEZE_GPT2_BASE = True

print("Output dir:", OUTPUT_DIR)
print("Final training:", EPOCHS, "epochs | early-stop patience", EARLY_STOP_PATIENCE)
print("Batches/epoch:", "full train set" if FULL_TRAIN_MAX_BATCHES is None else FULL_TRAIN_MAX_BATCHES)
print("Batch size:", BATCH_SIZE)
print(f"ViT+GPT-2: lr={LEARNING_RATE:g}, wd={WEIGHT_DECAY}, dropout={DROPOUT}, freeze_base={FREEZE_GPT2_BASE}")

## 3. Download MS-COCO data

Pure-Python download/extract (`urllib` + `zipfile`), so it runs on Colab or locally. Existence checks skip the work if the data is already present.

In [ ]:
import urllib.request, zipfile

ANNOTATIONS_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
IMAGES_URL = ("http://images.cocodataset.org/zips/val2017.zip" if COCO_SPLIT == "val"
              else "http://images.cocodataset.org/zips/train2017.zip")

ann_dir = os.path.join(DATA_DIR, "annotations")
img_dir = os.path.join(DATA_DIR, f"{COCO_SPLIT}2017")

def _download_and_extract(url, zip_path, extract_to):
    def _progress(block_num, block_size, total_size):
        if total_size > 0:
            pct = min(100, block_num * block_size * 100 / total_size)
            print(f"\r  downloading... {pct:5.1f}%", end="")
    print("Downloading", url)
    urllib.request.urlretrieve(url, zip_path, _progress)
    print("\n  extracting...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_to)
    os.remove(zip_path)
    print("  done.")

if not os.path.exists(ann_dir):
    _download_and_extract(ANNOTATIONS_URL, os.path.join(DATA_DIR, "annotations.zip"), DATA_DIR)
else:
    print("Annotations already extracted.")

if not os.path.exists(img_dir):
    _download_and_extract(IMAGES_URL, os.path.join(DATA_DIR, f"{COCO_SPLIT}2017.zip"), DATA_DIR)
else:
    print(f"{COCO_SPLIT}2017 images already extracted.")

## 4. Load captions and split by image id

Leakage-safe split by **image id** so an image never appears in both train and validation.

In [ ]:
annotations_file = os.path.join(DATA_DIR, "annotations", f"captions_{COCO_SPLIT}2017.json")
with open(annotations_file, "r") as f:
    coco_data = json.load(f)

img_id_to_filename = {img["id"]: img["file_name"] for img in coco_data["images"]}
img_id_to_captions = defaultdict(list)
for ann in coco_data["annotations"]:
    img_id_to_captions[ann["image_id"]].append(ann["caption"])

def make_image_id_split(image_ids, train_fraction=0.90, seed=42):
    image_ids = list(image_ids)
    rng = random.Random(seed)
    rng.shuffle(image_ids)
    split_idx = int(train_fraction * len(image_ids))
    return set(image_ids[:split_idx]), set(image_ids[split_idx:])

train_img_ids, val_img_ids = make_image_id_split(img_id_to_filename.keys(), 0.90, SEED)
train_annotations = [a for a in coco_data["annotations"] if a["image_id"] in train_img_ids]
val_annotations = [a for a in coco_data["annotations"] if a["image_id"] in val_img_ids]

assert train_img_ids.isdisjoint(val_img_ids)
print("Train images:", len(train_img_ids), " Val images:", len(val_img_ids))
print("Train captions:", len(train_annotations), " Val captions:", len(val_annotations))

## 5. Dataset

Returns the raw PIL image, the caption string, and the image id. Image preprocessing and text encoding happen in `collate`.

In [ ]:
class CocoCaptionDataset(Dataset):
    """Returns (PIL image, caption string, image_id)."""
    def __init__(self, img_dir, annotations, img_id_to_filename):
        self.img_dir = img_dir
        self.annotations = list(annotations)
        self.img_id_to_filename = dict(img_id_to_filename)

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        ann = self.annotations[idx]
        img_id = ann["image_id"]
        path = os.path.join(self.img_dir, self.img_id_to_filename[img_id])
        image = Image.open(path).convert("RGB")
        return image, ann["caption"], img_id

train_dataset = CocoCaptionDataset(img_dir, train_annotations, img_id_to_filename)
val_dataset = CocoCaptionDataset(img_dir, val_annotations, img_id_to_filename)
print("Datasets ready:", len(train_dataset), "train rows,", len(val_dataset), "val rows")

## 6. ViT + GPT-2 model

The ViT encoder produces a sequence of patch features `(B, T, 768)`; the GPT-2 decoder attends to them via cross-attention. GPT-2 has no pad token (pad==eos collision), so a dedicated `<|pad|>` token is added. GPT-2's `generate()` does not forward `encoder_hidden_states`, so generation uses a manual greedy loop with KV-caching that passes the encoder states every step.

In [ ]:
def preprocess_images(images, image_processor):
    return image_processor(list(images), return_tensors="pt").pixel_values


class ImageEncoder(nn.Module):
    """ViT encoder: image -> sequence of patch features (B, T, 768)."""
    def __init__(self, name):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(name)
        self.feat_dim = self.backbone.config.hidden_size

    def forward(self, images):
        out = self.backbone(pixel_values=images)
        return out.last_hidden_state


class GPT2Decoder(nn.Module):
    def __init__(self, feat_dim, tokenizer, dropout=0.1, freeze_base=False):
        super().__init__()
        cfg = AutoConfig.from_pretrained("gpt2")
        cfg.is_decoder = True
        cfg.add_cross_attention = True
        cfg.resid_pdrop = dropout
        cfg.embd_pdrop = dropout
        cfg.attn_pdrop = dropout
        self.gpt2 = AutoModelForCausalLM.from_pretrained("gpt2", config=cfg)
        self.gpt2.resize_token_embeddings(len(tokenizer))   # dedicated pad token
        self.enc_proj = nn.Linear(feat_dim, cfg.n_embd)
        self.tokenizer = tokenizer
        self.bos_id = tokenizer.bos_token_id
        self.eos_id = tokenizer.eos_token_id
        self.pad_id = tokenizer.pad_token_id
        if freeze_base:
            # Train only the (random-init) cross-attention + enc_proj; freeze the
            # rest of the pretrained GPT-2.
            for n, p in self.gpt2.named_parameters():
                p.requires_grad = ("crossattention" in n) or ("ln_cross_attn" in n)

    def forward(self, enc_seq, token_ids):
        enc_hidden = self.enc_proj(enc_seq)
        attn = (token_ids != self.pad_id).long()
        labels = token_ids.clone()
        labels[token_ids == self.pad_id] = -100
        out = self.gpt2(input_ids=token_ids, attention_mask=attn,
                        encoder_hidden_states=enc_hidden, labels=labels)
        return out.loss

    @torch.no_grad()
    def generate(self, enc_seq, max_len):
        enc_hidden = self.enc_proj(enc_seq)
        B = enc_hidden.size(0)
        dev = enc_hidden.device
        cur = torch.full((B, 1), self.bos_id, dtype=torch.long, device=dev)
        seqs = [[] for _ in range(B)]
        done = torch.zeros(B, dtype=torch.bool, device=dev)
        past = None
        for _ in range(max_len):
            out = self.gpt2(input_ids=cur, encoder_hidden_states=enc_hidden,
                            past_key_values=past, use_cache=True)
            past = out.past_key_values
            nxt = out.logits[:, -1, :].argmax(-1)
            for i in range(B):
                if done[i]:
                    continue
                tid = int(nxt[i].item())
                if tid == self.eos_id:
                    done[i] = True
                else:
                    seqs[i].append(tid)
            if bool(done.all()):
                break
            cur = nxt.unsqueeze(1)
        return seqs

    def decode(self, ids):
        return self.tokenizer.decode(ids, skip_special_tokens=True).strip()


class CaptioningModel(nn.Module):
    def __init__(self, encoder, decoder, freeze_encoder=True):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.freeze_encoder = freeze_encoder

    def _encode(self, images):
        if self.freeze_encoder:
            with torch.no_grad():
                return self.encoder(images)
        return self.encoder(images)

    def forward(self, images, labels):
        return self.decoder(self._encode(images), labels)

    @torch.no_grad()
    def generate(self, images, max_len):
        return self.decoder.generate(self._encode(images), max_len)

    def decode(self, ids):
        return self.decoder.decode(ids)

## 7. Config, builder, training, and BLEU evaluation

`run_experiment` runs the shared training loop with **per-epoch best-BLEU checkpointing, early stopping, and rollback** (identical to the comparison notebook) and returns the per-epoch history for plotting.

In [ ]:
@dataclass
class ExperimentConfig:
    name: str
    encoder_name: str
    decoder: str = "gpt2"
    learning_rate: float = 1e-4
    weight_decay: float = 0.0
    dropout: float = 0.1
    freeze_gpt2_base: bool = False
    batch_size: int = 32
    epochs: int = 15
    max_train_batches: Optional[int] = None   # None = use the full train set each epoch


def build_model(config):
    encoder = ImageEncoder(config.encoder_name)
    feat_dim = encoder.feat_dim
    tok = AutoTokenizer.from_pretrained("gpt2")
    if tok.pad_token is None:
        tok.add_special_tokens({"pad_token": "<|pad|>"})
    decoder = GPT2Decoder(feat_dim, tok, dropout=config.dropout,
                          freeze_base=config.freeze_gpt2_base)
    model = CaptioningModel(encoder, decoder, freeze_encoder=True)
    for p in model.encoder.parameters():
        p.requires_grad = False
    image_processor = AutoImageProcessor.from_pretrained(config.encoder_name)
    return {"model": model, "image_processor": image_processor}


def make_collate(bundle):
    image_processor = bundle["image_processor"]
    tok = bundle["model"].decoder.tokenizer

    def collate(batch):
        images, captions, image_ids = zip(*batch)
        pixel_values = preprocess_images(images, image_processor)
        enc = tok(list(captions), add_special_tokens=False,
                  truncation=True, max_length=MAX_TEXT_LEN - 2)
        seqs = [[tok.bos_token_id] + ids + [tok.eos_token_id] for ids in enc["input_ids"]]
        maxlen = max(len(s) for s in seqs)
        labels = torch.full((len(seqs), maxlen), tok.pad_token_id, dtype=torch.long)
        for i, s in enumerate(seqs):
            labels[i, :len(s)] = torch.tensor(s, dtype=torch.long)
        return pixel_values, labels, torch.tensor(image_ids, dtype=torch.long)
    return collate


def train_one_epoch(model, loader, optimizer, device, max_batches=None):
    model.train()
    total_loss, n = 0.0, 0
    for batch_idx, (pixel_values, labels, _) in enumerate(loader):
        if max_batches is not None and batch_idx >= max_batches:
            break
        pixel_values = pixel_values.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        loss = model(pixel_values, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], 1.0)
        optimizer.step()
        total_loss += float(loss.item()); n += 1
    return total_loss / max(n, 1)


def _tok(text):
    return word_tokenize(text.lower())


@torch.no_grad()
def evaluate_bleu(model, loader, device, num_batches=None):
    model.eval()
    references, hypotheses, seen = [], [], set()
    smoothing = SmoothingFunction().method1
    for batch_idx, (pixel_values, labels, image_ids) in enumerate(loader):
        if num_batches is not None and batch_idx >= num_batches:
            break
        pixel_values = pixel_values.to(device)
        gen_ids = model.generate(pixel_values, MAX_GEN_LEN)
        for j in range(pixel_values.size(0)):
            img_id = int(image_ids[j].item())
            if img_id in seen:
                continue
            seen.add(img_id)
            hypotheses.append(_tok(model.decode(gen_ids[j])))
            references.append([_tok(c) for c in img_id_to_captions[img_id]])
    return 0.0 if not hypotheses else corpus_bleu(references, hypotheses, smoothing_function=smoothing)


def run_experiment(config, save_ckpt=True, verbose=True):
    """Shared loop: per-epoch best-BLEU checkpoint, early stopping, rollback.
    Returns (result_dict, bundle, history)."""
    if verbose:
        print("\n" + "=" * 80); print("Running:", config.name); print("=" * 80)

    bundle = build_model(config)
    model = bundle["model"].to(device)
    collate = make_collate(bundle)
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True,
                              num_workers=NUM_WORKERS, collate_fn=collate,
                              pin_memory=torch.cuda.is_available())
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False,
                            num_workers=NUM_WORKERS, collate_fn=collate,
                            pin_memory=torch.cuda.is_available())

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=config.learning_rate,
                                  weight_decay=config.weight_decay)

    ckpt_path = os.path.join(OUTPUT_DIR, f"{config.name}.pt")
    losses, val_bleus = [], []
    best_bleu, best_epoch, best_state = -1.0, -1, None
    epochs_no_improve = 0
    start = time.time()

    for epoch in range(config.epochs):
        loss = train_one_epoch(model, train_loader, optimizer, device,
                               max_batches=config.max_train_batches)
        losses.append(loss)
        val_bleu = evaluate_bleu(model, val_loader, device, num_batches=EVAL_NUM_BATCHES)
        val_bleus.append(val_bleu)

        improved = val_bleu > best_bleu + EARLY_STOP_MIN_DELTA
        flag = ""
        if improved:
            best_bleu, best_epoch = val_bleu, epoch
            epochs_no_improve = 0
            best_state = copy.deepcopy(model.state_dict())
            if save_ckpt:
                torch.save({"state_dict": best_state, "config": asdict(config),
                            "bleu4": best_bleu, "epoch": epoch + 1}, ckpt_path)
            flag = "  <-- best so far" + (" (saved)" if save_ckpt else "")
        else:
            epochs_no_improve += 1

        if verbose:
            print(f"Epoch {epoch+1}/{config.epochs}: train loss = {loss:.4f}, "
                  f"val BLEU-4 = {val_bleu:.4f}{flag}")

        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            if verbose:
                print(f"Early stopping: no improvement for {EARLY_STOP_PATIENCE} "
                      f"epoch(s). Best was epoch {best_epoch+1} (BLEU-4 = {best_bleu:.4f}).")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    result = {
        "name": config.name,
        "encoder": config.encoder_name,
        "decoder": config.decoder,
        "learning_rate": config.learning_rate,
        "weight_decay": config.weight_decay,
        "dropout": config.dropout,
        "freeze_gpt2_base": config.freeze_gpt2_base,
        "best_epoch": best_epoch + 1,
        "epochs_trained": len(losses),
        "final_train_loss": losses[-1] if losses else None,
        "bleu4": best_bleu,
        "train_time_seconds": round(time.time() - start, 2),
        "trainable_params": sum(p.numel() for p in params),
        "checkpoint": ckpt_path if save_ckpt else "",
    }
    history = {"losses": losses, "val_bleus": val_bleus}
    return result, bundle, history

## 8. Train ViT + GPT-2 (full set, 15 epochs, early stopping)

The best config from the comparison, retrained on the full training set with early stopping and per-epoch best-weight checkpointing.

In [ ]:
config = ExperimentConfig(
    name="vit_gpt2_full",
    encoder_name=ENCODER_NAME,
    decoder="gpt2",
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    dropout=DROPOUT,
    freeze_gpt2_base=FREEZE_GPT2_BASE,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    max_train_batches=FULL_TRAIN_MAX_BATCHES,
)

result, bundle, history = run_experiment(config, save_ckpt=True)

results_df = pd.DataFrame([result])
results_csv = os.path.join(OUTPUT_DIR, "vit_gpt2_training_results.csv")
results_df.to_csv(results_csv, index=False)
print("\nSaved results:", results_csv)
print("Best BLEU-4:", round(result["bleu4"], 4), "at epoch", result["best_epoch"])
results_df

## 9. Visualizations

The same two plots as the fine-tune comparison notebook: validation BLEU-4 per epoch and training loss per epoch.

In [ ]:
DISPLAY_NAME = "ViT + GPT-2"

# ---- Validation BLEU-4 ------------------------------------------------------
plt.figure(figsize=(9, 6))
epochs_axis = range(1, len(history["val_bleus"]) + 1)
plt.plot(epochs_axis, history["val_bleus"], marker="o", linewidth=2, label=DISPLAY_NAME)
plt.title("Validation BLEU-4 over training (15 epochs, early stopping)")
plt.xlabel("Epoch"); plt.ylabel("Validation BLEU-4")
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
bleu_plot = os.path.join(OUTPUT_DIR, "bleu4_vit_gpt2.png")
plt.savefig(bleu_plot, dpi=150); plt.show()
print("Saved:", bleu_plot)

# ---- Training loss ---------------------------------------------------------
plt.figure(figsize=(9, 6))
epochs_axis = range(1, len(history["losses"]) + 1)
plt.plot(epochs_axis, history["losses"], marker="s", linewidth=2, label=DISPLAY_NAME)
plt.title("Training loss over training")
plt.xlabel("Epoch"); plt.ylabel("Train loss")
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
loss_plot = os.path.join(OUTPUT_DIR, "loss_vit_gpt2.png")
plt.savefig(loss_plot, dpi=150); plt.show()
print("Saved:", loss_plot)

## 10. Save the best model to Google Drive (persistent storage)

`OUTPUT_DIR` is on the ephemeral Colab VM disk. This copies the best checkpoint, the result CSV, and the plots to Drive. On a local Jupyter it just reports the local path.

In [ ]:
import shutil

DRIVE_SAVE_DIR = "/content/drive/MyDrive/image_captioning_vit_gpt2"

try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
    best_ckpt = result["checkpoint"]
    for fpath in [best_ckpt, results_csv, bleu_plot, loss_plot]:
        if fpath and os.path.exists(fpath):
            shutil.copy2(fpath, os.path.join(DRIVE_SAVE_DIR, os.path.basename(fpath)))
    print("Saved best checkpoint + CSV + plots to Drive:", DRIVE_SAVE_DIR)
except ImportError:
    print("Not on Colab — everything already persists locally under:", os.path.abspath(OUTPUT_DIR))

## 11. Report-ready notes

- **What this run does:** trains the comparison's winning architecture (**ViT + GPT-2**) with its best hyperparameters (`lr=1e-4, weight_decay=0.01, dropout=0.2, freeze_gpt2_base=True`) on the **full training set** for **15 epochs with early stopping** (monitor val BLEU-4, patience 3), keeping the per-epoch best weights and rolling back to them.
- **Frozen encoder + partial decoder:** the ViT encoder is a frozen feature extractor; with `freeze_gpt2_base=True` only the GPT-2 cross-attention layers and the encoder projection are trained.
- **Outputs:** `vit_gpt2_training_results.csv`, `bleu4_vit_gpt2.png`, `loss_vit_gpt2.png`, and a `vit_gpt2_full.pt` checkpoint (best epoch).